# Garment Production Intelligence

## 01 - Data Understanding

**Objective:** Explore the garment manufacturing
dataset and understand its structure.

**Dataset:** Productivity Prediction of
Garment Employees (UCI Repository).

In [1]:

import pandas as pd
import numpy as np

print("Garment Production Intelligence")
print("Python environment is working!")
print("Pandas version:", pd.__version__)


Garment Production Intelligence
Python environment is working!
Pandas version: 3.1.1


In [2]:

from pathlib import Path

# Find the project folder
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Locate the original CSV file
file_path = project_dir / "data" / "raw" / "garments_worker_productivity.csv"

# Load the dataset
df = pd.read_csv(file_path)

# Display the first five records
df.head()


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [3]:

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])


Total rows: 1197
Total columns: 15


In [4]:

# Check column names, data types and non-null counts
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

In [5]:

# Count missing values in each column
missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)


Missing values by column:
date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


In [6]:

# Calculate missing value percentages
missing_percentage = (
    df.isnull().sum() / len(df)
) * 100

missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": missing_percentage.round(2)
})

missing_report


,Missing Count,Missing Percentage
date,0,0.00
quarter,0,0.00
department,0,0.00
day,0,0.00
team,0,0.00
targeted_productivity,0,0.00
smv,0,0.00
wip,506,42.27
over_time,0,0.00
incentive,0,0.00


In [7]:

# Investigate missing WIP values by department

wip_by_department = df.groupby("department").agg(
    total_records=("wip", "size"),
    missing_wip=("wip", lambda x: x.isna().sum())
)

wip_by_department["missing_percentage"] = (
    wip_by_department["missing_wip"]
    / wip_by_department["total_records"] * 100
).round(2)

wip_by_department


,total_records,missing_wip,missing_percentage
department,,,
finishing,249,249,100.0
finishing,257,257,100.0
sweing,691,0,0.0


In [8]:

# Check duplicate rows

duplicate_count = df.duplicated().sum()

print("Total duplicate rows:", duplicate_count)


Total duplicate rows: 0


In [9]:
# Display exact department names
for department in df["department"].unique():
    print(repr(department))


'sweing'
'finishing '
'finishing'


In [10]:
# Create a working copy to preserve the original data
df_clean = df.copy()

# Remove extra spaces and standardize case
df_clean["department"] = (
    df_clean["department"]
    .str.strip()
    .str.lower()
)

# Display department counts
print(df_clean["department"].value_counts())


department
sweing       691
finishing    506
Name: count, dtype: int64


In [11]:
# Missing WIP values after department name cleaning
wip_summary = df_clean.groupby("department").agg(
    total_records=("wip", "size"),
    missing_wip=("wip", lambda x: x.isna().sum())
)

wip_summary["missing_percentage"] = (
    wip_summary["missing_wip"]
    / wip_summary["total_records"] * 100
).round(2)

wip_summary


,total_records,missing_wip,missing_percentage
department,,,
finishing,506,506,100.0
sweing,691,0,0.0


In [12]:
# Check descriptive statistics for productivity

productivity_summary = df_clean[
    ["targeted_productivity", "actual_productivity"]
].describe()

productivity_summary


,targeted_productivity,actual_productivity
count,1197.000000,1197.000000
mean,0.729632,0.735091
std,0.097891,0.174488
min,0.070000,0.233705
25%,0.700000,0.650307
50%,0.750000,0.773333
75%,0.800000,0.850253
max,0.800000,1.120437


In [13]:
# Investigate productivity values outside the expected 0-1 range

for column in ["targeted_productivity", "actual_productivity"]:

    unusual_values = df_clean[
        (df_clean[column] < 0) |
        (df_clean[column] > 1)
    ]

    print(f"{column}: {len(unusual_values)} unusual records")


targeted_productivity: 0 unusual records
actual_productivity: 37 unusual records
